In [4]:
from dataclasses import dataclass, field
from typing import List, Optional
from collections import defaultdict, deque
from ucimlrepo import fetch_ucirepo 
from sklearn.model_selection import train_test_split
import random
import time
import pandas as pd

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)



@dataclass
class AttributeRecord:
    rid: int
    value: float
    classLabel: str


@dataclass
class AttributeList:
    attribute_id: int
    records: List[AttributeRecord] = field(default_factory=list)


@dataclass
class SplitResult:
    gini: float
    index: int
    value: float
    attribute_id: int



class Node:
    def __init__(self):
        self.record_ids: set = set()
        self.split_attribute_id: Optional[int] = None
        self.split_threshold: Optional[float] = None
        self.class_label: Optional[str] = None
        self.is_leaf: bool = False
        self.left_child: Optional["Node"] = None
        self.right_child: Optional["Node"] = None

    def __repr__(self):
        if self.is_leaf:
            return f"Leaf(label={self.class_label}, n={len(self.record_ids)})"
        return (f"Node(split=attr_{self.split_attribute_id}<={self.split_threshold:.4f}, "
                f"n={len(self.record_ids)})")


class Sprint:
    def __init__(self, min_samples_split: int = 2):
        self.attribute_lists: List[AttributeList] = []
        self.root: Node = Node()
        self.min_samples_split = min_samples_split


    def parse_dataset(self, X, Y):
        Y = Y.squeeze()  # ensure Series, not DataFrame
        self.attribute_lists = []
        for attr_id, attr in enumerate(X.columns):
            al = AttributeList(attribute_id=attr_id)
            al.records = [
                AttributeRecord(rid, value, Y.iloc[rid])
                for rid, value in enumerate(X[attr])
            ]
            self.attribute_lists.append(al)
        self.root.record_ids = set(range(len(Y)))

    def sort_attribute_lists(self):

        for al in self.attribute_lists:
            al.records.sort(key=lambda r: r.value)


    @staticmethod
    def calculate_class_histogram(records) -> dict:

        hist = defaultdict(int)
        for r in records:
            hist[r.classLabel] += 1
        return hist

    @staticmethod
    def gini(hist: dict) -> float:

        total = sum(hist.values())
        if total == 0:
            return 0.0
        return 1.0 - sum((c / total) ** 2 for c in hist.values())


    def calculate_gini(self, attribute_list: AttributeList, record_ids: set) -> SplitResult:
        records = [r for r in attribute_list.records if r.rid in record_ids]

        n = len(records)
        if n < 2:
            return SplitResult(float("inf"), 0, 0.0, attribute_list.attribute_id)

        left_hist = self.calculate_class_histogram(records)
        right_hist: dict = defaultdict(int)

        left_size = n
        right_size = 0

        best_gini = float("inf")
        best_index = -1

        for i in range(n - 1):
            record = records[i]

            left_hist[record.classLabel] -= 1
            if left_hist[record.classLabel] == 0:
                del left_hist[record.classLabel]
            right_hist[record.classLabel] += 1

            left_size -= 1
            right_size += 1

            if records[i].value == records[i + 1].value:
                continue

            g_left = self.gini(left_hist)
            g_right = self.gini(right_hist)
            split_gini = (left_size / n) * g_left + (right_size / n) * g_right

            if split_gini < best_gini:
                best_gini = split_gini
                best_index = i

        if best_index == -1:
            return SplitResult(float("inf"), 0, 0.0, attribute_list.attribute_id)

        split_value = (records[best_index].value + records[best_index + 1].value) / 2
        return SplitResult(best_gini, best_index, split_value, attribute_list.attribute_id)



    def find_best_split(self, node: Node) -> SplitResult:
        best = SplitResult(float("inf"), 0, 0.0, -1)
        for al in self.attribute_lists:
            result = self.calculate_gini(al, node.record_ids)
            if result.gini < best.gini:
                best = result
        return best

    def is_pure(self, record_ids: set) -> bool:

        al = self.attribute_lists[0]
        records = [r for r in al.records if r.rid in record_ids]
        hist = self.calculate_class_histogram(records)
        return len(hist) <= 1

    def majority_label(self, record_ids: set) -> str:
        al = self.attribute_lists[0]
        records = [r for r in al.records if r.rid in record_ids]
        hist = self.calculate_class_histogram(records)
        return max(hist, key=hist.__getitem__)

    def split_node(self, node: Node):
        split = self.find_best_split(node)

        if split.gini == float("inf"):
            node.is_leaf = True
            node.class_label = self.majority_label(node.record_ids)
            return None, None

        node.split_attribute_id = split.attribute_id
        node.split_threshold = split.value

        left = Node()
        right = Node()

        attr_list = self.attribute_lists[split.attribute_id]

        for record in attr_list.records:
            if record.rid not in node.record_ids:
                continue
            if record.value <= split.value:
                left.record_ids.add(record.rid)
            else:
                right.record_ids.add(record.rid)

        node.left_child = left
        node.right_child = right
        return left, right


    def build_tree(self):
        self.sort_attribute_lists()

        queue = deque([self.root])

        while queue:
            node = queue.popleft()

            if (len(node.record_ids) < self.min_samples_split
                    or self.is_pure(node.record_ids)):
                node.is_leaf = True
                node.class_label = self.majority_label(node.record_ids)
                continue

            left, right = self.split_node(node)

            if left is None:
                continue

            if left.record_ids:
                queue.append(left)
            else:
                left.is_leaf = True
                left.class_label = self.majority_label(node.record_ids)

            if right.record_ids:
                queue.append(right)
            else:
                right.is_leaf = True
                right.class_label = self.majority_label(node.record_ids)

 

    def predict_one(self, values: list) -> str:
        node = self.root
        while not node.is_leaf:
            if values[node.split_attribute_id] <= node.split_threshold:
                node = node.left_child
            else:
                node = node.right_child
        return node.class_label

    def predict(self, X) -> List[str]:
        return [
            self.predict_one(X.iloc[i].tolist())
            for i in range(len(X))
        ]


    def print_tree(self, node: Optional[Node] = None, depth: int = 0):
        if node is None:
            node = self.root
        indent = "  " * depth
        if node.is_leaf:
            print(f"{indent}→ class={node.class_label}  (n={len(node.record_ids)})")
        else:
            print(f"{indent}[attr_{node.split_attribute_id} <= {node.split_threshold:.4f}]"
                  f"  (n={len(node.record_ids)})")
            self.print_tree(node.left_child, depth + 1)
            self.print_tree(node.right_child, depth + 1)



In [5]:
repo_to_id = dict()
repo_to_id["iris"] = 53
repo_to_id["wine"] = 186
repo_to_id["cancer"] = 17
repo_to_id["musk"] = 75


In [6]:
import random

def load_dataset(dataset_name: str):
    if dataset_name in repo_to_id:
        dataset = fetch_ucirepo(id=repo_to_id[dataset_name])
        X = dataset.data.features
        y = dataset.data.targets
        print("Dataset loaded:", dataset_name)
    else:
        print("No dataset named:", dataset_name)
        return

    # if dataset_name == "wine":
    #     print("WINE")

    #     def label(q):
    #         if q <= 5:
    #             return "low"
    #         elif q <= 6:
    #             return "medium"
    #         else:
    #             return "high"

    #     y["quality"] = y["quality"].apply(label)

    print(y)
    return X, y

def test_dataset(X,y):
    X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random.randint(0, 10_000),stratify = y
    )
    s=Sprint()
    s.parse_dataset(X_train,y_train)
    s.build_tree()
    y_pred = s.predict(X_test)
    y_true = y_test.squeeze().tolist()
    print(classification_report(y_true, y_pred))

In [7]:



def benchmark_datasets(
    datasets,
    min_samples_values,
    n_runs=10,
    output_file="benchmark_results.csv"
):
    results = []

    for dataset in datasets:

        X, y = load_dataset(dataset)

        for min_samples_split in min_samples_values:

            build_times = []
            predict_times = []

            accuracies = []
            precisions = []
            recalls = []
            f1_scores = []

            for run in range(n_runs):

                X_train, X_test, y_train, y_test = train_test_split(
                    X,
                    y,
                    test_size=0.2,
                    random_state=random.randint(0, 10_000),
                    stratify=y
                )

                tree = Sprint(min_samples_split=min_samples_split)
                tree.parse_dataset(X_train, y_train)

                start = time.perf_counter()
                tree.build_tree()
                build_time = time.perf_counter() - start

                start = time.perf_counter()
                y_pred = tree.predict(X_test)
                predict_time = time.perf_counter() - start

                y_true = y_test.squeeze().tolist()

                build_times.append(build_time)
                predict_times.append(predict_time)

                accuracies.append(
                    accuracy_score(y_true, y_pred)
                )

                precisions.append(
                    precision_score(
                        y_true,
                        y_pred,
                        average="weighted",
                        zero_division=0
                    )
                )

                recalls.append(
                    recall_score(
                        y_true,
                        y_pred,
                        average="weighted",
                        zero_division=0
                    )
                )

                f1_scores.append(
                    f1_score(
                        y_true,
                        y_pred,
                        average="weighted",
                        zero_division=0
                    )
                )

            results.append({
                "dataset": dataset,
                "min_samples_split": min_samples_split,
                "runs": n_runs,
                "avg_build_time_s": sum(build_times) / n_runs,
                "avg_predict_time_s": sum(predict_times) / n_runs,
                "avg_accuracy": sum(accuracies) / n_runs,
                "avg_precision": sum(precisions) / n_runs,
                "avg_recall": sum(recalls) / n_runs,
                "avg_f1": sum(f1_scores) / n_runs,
            })

    df = pd.DataFrame(results)

    print(df)

    df.to_csv(output_file, index=False)
    print(f"\nResults saved to {output_file}")


In [9]:

benchmark_datasets(
    ["iris","musk","cancer","wine"],
    min_samples_values=[2, 5, 10, 20, 50],
    n_runs=10
)

Dataset loaded: iris
              class
0       Iris-setosa
1       Iris-setosa
2       Iris-setosa
3       Iris-setosa
4       Iris-setosa
..              ...
145  Iris-virginica
146  Iris-virginica
147  Iris-virginica
148  Iris-virginica
149  Iris-virginica

[150 rows x 1 columns]
Dataset loaded: musk
      class
0       1.0
1       1.0
2       1.0
3       1.0
4       1.0
...     ...
6593    0.0
6594    0.0
6595    0.0
6596    0.0
6597    0.0

[6598 rows x 1 columns]
Dataset loaded: cancer
    Diagnosis
0           M
1           M
2           M
3           M
4           M
..        ...
564         M
565         M
566         M
567         M
568         B

[569 rows x 1 columns]
Dataset loaded: wine
      quality
0           5
1           5
2           5
3           6
4           5
...       ...
6492        6
6493        5
6494        6
6495        7
6496        6

[6497 rows x 1 columns]
   dataset  min_samples_split  runs  avg_build_time_s  avg_predict_time_s  \
0     iris         

In [6]:
import pandas as pd


def format_csv_floats(input_file, output_file):
    df = pd.read_csv(input_file)

    numeric_cols = df.select_dtypes(include=["float", "float64", "float32"]).columns

    # format as strings with scientific notation if needed
    for col in numeric_cols:
        df[col] = df[col].apply(lambda x: f"{x:.4g}")  # keeps small numbers visible

    df.to_csv(output_file, index=False)

In [7]:
format_csv_floats("benchmark_results.csv", "benchmark_results_rounded.csv")